In [1]:
!pip install sentence-transformers -q

In [2]:
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
jobs_data = pd.read_csv("final_jobs_dataset (1).csv")
cv_data = pd.read_csv("cv_cleaned.csv")

cv_texts = cv_data['final_text'].fillna("").astype(str).tolist()
job_texts = jobs_data['job_description'].fillna("").astype(str).tolist()

print("Data Loaded ✅")
print("CVs:", len(cv_texts))
print("Jobs:", len(job_texts))

Data Loaded ✅
CVs: 3500
Jobs: 3685


In [5]:
print("CV Sample:")
print(cv_texts[0][:300])

print("\nJob Sample:")
print(job_texts[0][:300])

CV Sample:
jessica claire montgomery street san francisco ca resumesampleexamplecom professional summary highly skilled software development professional bringing years software design development integration advanced knowledge java skills agile html xml jdbc tomcat work history senior java developertech lead 

Job Sample:
about hopper

at hopper, we’re on a mission to make booking travel faster, easier, and more transparent. we are leveraging the power that comes from combining massive amounts of data and machine learning to build the world’s fastest-growing travel app -- one that enables our customers to save money 


In [6]:
bert_model = SentenceTransformer('all-MiniLM-L6-v2')

d:\New folder (6)\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP d

In [7]:
cv_embeddings_bert = bert_model.encode(
    cv_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

job_embeddings_bert = bert_model.encode(
    job_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches: 100%|██████████| 116/116 [01:02<00:00,  1.86it/s]


In [8]:
print("CV Embeddings Shape:", cv_embeddings_bert.shape)
print("Job Embeddings Shape:", job_embeddings_bert.shape)

CV Embeddings Shape: (3500, 384)
Job Embeddings Shape: (3685, 384)


In [9]:
similarity_matrix_bert = cosine_similarity(
    cv_embeddings_bert,
    job_embeddings_bert
)

In [10]:
print("Similarity Matrix Shape:", similarity_matrix_bert.shape)
print("Higher score = better semantic match between CV and Job Description")

Similarity Matrix Shape: (3500, 3685)
Higher score = better semantic match between CV and Job Description


In [11]:
for i in range(3):
    top_jobs = np.argsort(similarity_matrix_bert[i])[::-1][:5]

    print(f"\nCV {i} recommendations:")

    for j in top_jobs:
        print("Job:", jobs_data.iloc[j]['job_title'])
        print("Score:", similarity_matrix_bert[i][j])
        print("-"*30)


CV 0 recommendations:
Job: data modeler/data analyst
Score: 0.5950824
------------------------------
Job: senior data engineer
Score: 0.5731243
------------------------------
Job: it data engineer
Score: 0.5712783
------------------------------
Job: sr data analyst
Score: 0.5624286
------------------------------
Job: big data engineer
Score: 0.55635226
------------------------------

CV 1 recommendations:
Job: data scientist
Score: 0.5761539
------------------------------
Job: data modeler/data analyst
Score: 0.5400808
------------------------------
Job: data science & machine learning
Score: 0.53706676
------------------------------
Job: java with big data engineer
Score: 0.5138591
------------------------------
Job: informatica data modeler
Score: 0.5078175
------------------------------

CV 2 recommendations:
Job: data science & machine learning
Score: 0.65975577
------------------------------
Job: principal data engineer
Score: 0.62493575
------------------------------
Job: java w

In [12]:
print("Top-5 jobs retrieved successfully for each CV ✅")

Top-5 jobs retrieved successfully for each CV ✅


In [13]:
i = 0
top_jobs = np.argsort(similarity_matrix_bert[i])[::-1][:5]

df_results = pd.DataFrame({
    "Recommended Job": [jobs_data.iloc[j]['job_title'] for j in top_jobs],
    "Match Score": [f"{similarity_matrix_bert[i][j]*100:.2f}%" for j in top_jobs],
    "Description Preview": [jobs_data.iloc[j]['job_description'][:100] + "..." for j in top_jobs]
})

df_results

,Recommended Job,Match Score,Description Preview
0,data modeler/data analyst,59.51%,job description:\n\n· bs/ba degree with 6+ yea...
1,senior data engineer,57.31%,role: senior data engineer\n\njob type: contra...
2,it data engineer,57.13%,this customer is the world's largest manufactu...
3,sr data analyst,56.24%,my direct client in center city philadelphia i...
4,big data engineer,55.64%,title: software engineering - big data\nlocati...


In [14]:
top_5_scores = similarity_matrix_bert[i][top_jobs]
print("Average Top-5 Match Score:", top_5_scores.mean())

Average Top-5 Match Score: 0.5716532


In [15]:
np.save("cv_embeddings_bert.npy", cv_embeddings_bert)
np.save("job_embeddings_bert.npy", job_embeddings_bert)

print("Embeddings saved successfully ✅")

Embeddings saved successfully ✅


In [16]:
import numpy as np

cv_embeddings = np.load("cv_embeddings_bert.npy")
job_embeddings = np.load("job_embeddings_bert.npy")

In [17]:
print(cv_embeddings.shape)
print(job_embeddings.shape)

(3500, 384)
(3685, 384)


In [18]:
print(cv_embeddings[0])

[ 5.08641712e-02 -7.60697350e-02 -2.93945931e-02  1.54265175e-02
 -2.18714662e-02 -1.91713106e-02 -7.08138347e-02  1.12828715e-02
 -1.08519897e-01  2.46648081e-02 -3.85322906e-02 -2.08712779e-02
  3.90019454e-02 -7.21972734e-02  9.00117755e-02  3.45093422e-02
  6.08565547e-02 -4.45139557e-02  1.18823219e-02 -6.19954728e-02
 -1.53927803e-02 -2.18234677e-02 -5.92300855e-02 -5.94187453e-02
 -2.32127421e-02  2.30828617e-02 -2.00659651e-02  1.27888639e-02
 -7.64433667e-02 -6.23257756e-02 -9.66318250e-02  8.34942088e-02
  8.68145600e-02  8.11704844e-02  5.67089766e-02  8.33690539e-02
 -4.24849428e-02  7.22491816e-02  5.14466390e-02 -3.80317792e-02
 -1.58511668e-01 -1.15875741e-02 -5.50862141e-02 -1.92213953e-02
 -4.02420247e-03 -1.00622945e-01 -7.35876039e-02 -9.35739428e-02
  4.09036130e-02  3.30038480e-02 -9.15548205e-02 -2.57579274e-02
  8.26113075e-02 -2.30107037e-03 -1.31580131e-02  1.50555270e-02
  1.54551468e-03 -3.06973904e-02 -6.43768832e-02 -4.67204079e-02
 -4.46727872e-02  1.04356

In [19]:
print(cv_embeddings[0][:10])

[ 0.05086417 -0.07606973 -0.02939459  0.01542652 -0.02187147 -0.01917131
 -0.07081383  0.01128287 -0.1085199   0.02466481]


In [20]:
import pandas as pd

df_emb = pd.DataFrame(cv_embeddings)
df_emb.head()

,0,1,2,3,4,5,6,7,8,9,...,374,375,376,377,378,379,380,381,382,383
0,0.050864,-0.076070,-0.029395,0.015427,-0.021871,-0.019171,-0.070814,0.011283,-0.108520,0.024665,...,0.029229,0.098212,-0.022590,0.014902,0.050404,0.018322,0.051080,-0.035472,-0.045784,0.004056
1,0.048936,-0.044995,-0.028675,-0.069765,-0.027521,-0.046056,0.009065,0.044963,-0.077154,0.042421,...,0.042603,0.064457,0.004688,0.070380,0.049006,-0.012808,0.084342,0.000968,0.001448,0.020021
2,-0.048309,-0.041490,-0.007161,0.027420,-0.026790,-0.011672,-0.062471,-0.006510,-0.092823,0.040452,...,0.038938,0.123432,0.063805,-0.017433,0.099677,0.025767,0.044352,-0.065372,-0.014411,0.027591
3,0.011886,-0.071153,-0.027228,0.044289,-0.062909,-0.031058,-0.012409,-0.004627,-0.117843,-0.023648,...,0.025476,0.081046,0.017459,0.036306,0.054697,0.063012,0.065934,0.011802,-0.009865,0.002399
4,0.022853,-0.053448,-0.031811,0.020541,-0.057424,-0.037365,0.007573,0.013839,-0.118231,-0.005468,...,0.016906,0.106030,0.013072,0.075341,0.090544,0.041729,0.085790,0.022726,-0.042125,-0.003396
